# Dual-branch TA baseline

- LSTM: 12:00~14:00, 10-minute sequence, branch-specific 40 inputs
- Tabular: June~August 14:00 CatBoost + Ridge, branch-specific 40 inputs
- 2024 weight selection, 2025 evaluation, 2019~2025 deployment refit
- ASOS TA is used only as the label; direct year and ASOS lag features are excluded.


In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT_POINT = Path('/content/drive')
if not (MOUNT_POINT / 'MyDrive').exists():
    drive.mount(str(MOUNT_POINT))
else:
    print('Google Drive is already mounted.')


In [ ]:
!pip install -q catboost
import torch, catboost, sklearn
print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available())
print('catboost=', catboost.__version__, 'sklearn=', sklearn.__version__)


In [ ]:
import subprocess, sys
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
BRANCH = 'agent/shortterm-12to14-pipeline'
# v2 경로를 사용해 이전에 복제된 낡은 저장소 캐시와 충돌하지 않게 합니다.
REPO_DIR = Path('/content/SME_DATA_dualbranch_v2')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
SCRIPT_PATH = REPO_DIR / 'scripts/train_dualbranch_ta_baseline.py'
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('REPO_DIR=', REPO_DIR, 'commit=', commit)
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f'{SCRIPT_PATH} 파일이 GitHub 브랜치에 없습니다. 현재 commit={commit}, branch={BRANCH}'
    )
print('SCRIPT_PATH=', SCRIPT_PATH)


In [ ]:
BASE = MOUNT_POINT / 'MyDrive/SME_DATA/processed_station_features'
MASTER_CSV = BASE / 'final_train_dataset_19to25_master.csv'
SHORTTERM_CSV = BASE / 'shortterm_12to14_data/incremental_12to14_tables/shortterm_long_2019to2025.csv'
OUTPUT_DIR = BASE / 'model_experiments_19to25/dualbranch_ta_baseline'
for path in [MASTER_CSV, SHORTTERM_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)
print('MASTER_CSV=', MASTER_CSV)
print('SHORTTERM_CSV=', SHORTTERM_CSV)
print('OUTPUT_DIR=', OUTPUT_DIR)


In [ ]:
cmd = [
    sys.executable, '-u', str(SCRIPT_PATH),
    '--master-csv', str(MASTER_CSV),
    '--shortterm-long-csv', str(SHORTTERM_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'auto', '--threads', '4',
]
print('$', ' '.join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
returncode = process.wait()
if returncode:
    raise RuntimeError(f'Training failed (exit code {returncode})')


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUTPUT_DIR / 'metrics.csv'))
display(pd.read_csv(OUTPUT_DIR / 'ensemble_weights.csv'))
manifest = pd.read_csv(OUTPUT_DIR / 'branch_feature_manifest.csv')
display(manifest.groupby(['branch', 'priority']).size().rename('feature_count').reset_index())
display(pd.read_csv(OUTPUT_DIR / 'ridge_feature_importance.csv').head(20))
if (OUTPUT_DIR / 'catboost_feature_importance.csv').exists():
    display(pd.read_csv(OUTPUT_DIR / 'catboost_feature_importance.csv').head(20))


## Outputs

`evaluation_models/` contains models refit through 2024 for the honest 2025 check.
`deployment_models/` contains models refit on every 2019~2025 label for future inference.
